# NorthForge Finance — End-to-End Workflow Run

Runs the full business workflow through `WorkflowOrchestrator`: Foundry's Trial Balance
pipeline (staging → enrichment → reporting → posting → interface) followed by the GL
import of that pipeline's Interface output — all under a single `WorkflowRun`.

Each section below reads and displays the data actually persisted at that stage, straight
from the domain repositories (`TrialBalanceRepository` for Foundry, `GLRepository` for GL).

## Spark session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/27 20:09:59 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/27 20:09:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7bd74fb3-380a-4f44-80b0-eff227433311;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 72ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

## Imports and helpers

In [2]:
from dataclasses import asdict
from datetime import date

import pandas as pd
from pyspark.sql import functions as F

from core.logging import configure_logging
from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS,
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient

from registry import RegistryClient
from gl import GLClient
from gl.repository import GLRepository

from workflow import WorkflowOrchestrator


configure_logging()


def display_df(df):
    display(df.toPandas())


def display_records(records):
    # GLRepository reads (e.g. get_postings/get_rejections) return tuples
    # of dataclasses rather than Spark DataFrames.
    display(pd.DataFrame([asdict(record) for record in records]))

## Configure clients and build the orchestrator

`TrialBalancePipeline` only knows Foundry processing now — no `RunTracker` — and
`GLClient` only knows GL processing. `WorkflowOrchestrator` is the thin layer that owns
execution/workflow lifecycle across both and coordinates Foundry → GL.

In [3]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
)

registry = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)
gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry)

orchestrator = WorkflowOrchestrator(
    run_tracker=run_tracker,
    foundry_pipeline=pipeline,
    gl=gl,
)

## Run the full workflow (Foundry → GL)

`run_workflow()` creates a single `WorkflowRun`, executes the Foundry pipeline
(`STAGING → ENRICHMENT → REPORTING → POSTING → INTERFACE`), then runs `GL / IMPORT`
against that same workflow's `FOUNDRY / INTERFACE` output — all as one business workflow.

In [4]:
workflow_result = orchestrator.run_workflow()

business_dt = pipeline.config.business_dt

workflow_run_id = workflow_result.foundry.identity.workflow_run_id
gl_run_id = workflow_result.gl.producer_run_id

# Each Foundry zone's own execution identity (producer_run_id) — this is
# what inter-zone/read-layer selection uses now, not business_dt/batch_id.
staging_zone, enrichment_zone, reporting_zone, posting_zone, interface_zone = (
    workflow_result.foundry.zones
)
staging_producer_run_id = staging_zone.identity.run_id
enrichment_producer_run_id = enrichment_zone.identity.run_id
reporting_producer_run_id = reporting_zone.identity.run_id
posting_producer_run_id = posting_zone.identity.run_id
interface_producer_run_id = interface_zone.identity.run_id

2026-08-27 17:07:37,856 | INFO | workflow.orchestrator | Workflow started | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9 | dataclass=TRIAL_BALANCE | business_dt=2026-03-31
2026-08-27 17:07:37,860 | INFO | workflow.orchestrator | Foundry pipeline started | run_id=213cefb3-164b-4968-b651-4c3ec9e4e497 | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9
2026-08-27 17:07:37,862 | INFO | workflow.orchestrator | Foundry zone started | operation=STAGING | run_id=406335d9-a74b-4914-950d-b0e34191967f | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9
2026-08-27 17:07:44,407 | INFO | workflow.orchestrator | Foundry zone succeeded | operation=STAGING | run_id=406335d9-a74b-4914-950d-b0e34191967f | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9 | records=42
2026-08-27 17:07:44,413 | INFO | workflow.orchestrator | Foundry zone started | operation=ENRICHMENT | run_id=252c51a6-c44d-4898-ba88-009ce437ced4 | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9
2026-08-27 17:07:52,16

2026-08-27 17:08:00,655 | INFO | workflow.orchestrator | GL import succeeded | run_id=42a38c93-1d6a-4277-8bc2-d592d3b4da32 | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9 | received=7 | posted=7 | rejected=0 | source_producer_run_id=16c36d19-d910-4ae4-8553-937e2c1068f2
2026-08-27 17:08:00,658 | INFO | workflow.orchestrator | Workflow succeeded | workflow_run_id=231848ad-fe2b-48b3-9436-f560f7daa1c9
Foundry pipeline run complete: PipelineResult(identity=RunIdentity(workflow_run_id=UUID('231848ad-fe2b-48b3-9436-f560f7daa1c9'), run_id=UUID('213cefb3-164b-4968-b651-4c3ec9e4e497'), parent_run_id=None), status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, zones=(ZoneResult(identity=RunIdentity(workflow_run_id=UUID('231848ad-fe2b-48b3-9436-f560f7daa1c9'), run_id=UUID('406335d9-a74b-4914-950d-b0e34191967f'), parent_run_id=UUID('213cefb3-164b-4968-b651-4c3ec9e4e497')), zone='STAGING', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=42), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('

In [4]:
business_dt = '2026-03-31'
staging_producer_run_id = '406335d9-a74b-4914-950d-b0e34191967f'
enrichment_producer_run_id = '252c51a6-c44d-4898-ba88-009ce437ced4'
reporting_producer_run_id = 'acd7dfd7-903c-4291-bfe9-7447753bcbae'
posting_producer_run_id = '0cf97fc2-f09f-4de3-9907-6dcaeddec2bf'
interface_producer_run_id = '16c36d19-d910-4ae4-8553-937e2c1068f2'
gl_run_id = '42a38c93-1d6a-4277-8bc2-d592d3b4da32'

## Foundry persistence layers

Each Foundry zone is read straight from its own persisted table, selected by the
`producer_run_id` of the execution that produced it (via `TrialBalanceRepository`) — not
by `business_dt`/`batch_id`.

### Source

In [5]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,SRC_CLIENT_ID,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,80000.000000000000,USD
1,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,10000.000000000000,USD
2,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
3,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,90000.000000000000,USD
4,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_BACK_VALUED_ADJUSTMENT,USD,0E-12,USD
5,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_ADJUSTED_BALANCE,USD,90000.000000000000,USD
6,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,15000.000000000000,USD
7,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,5000.000000000000,USD
8,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
9,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,20000.000000000000,USD


### Staging

In [10]:
stg_df = repository.read_staging(staging_producer_run_id)

display_df(stg_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,10000.000000000000,1.000000000000,10000.000000000000,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,PREVIOUS_DAY_BALANCE,REPORTABLE,USD,80000.000000000000,1.000000000000,80000.000000000000,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,20000.000000000000,1.000000000000,20000.000000000000,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,5000.000000000000,1.000000000000,5000.000000000000,DEBIT,231848ad-fe2b-48b3-9436-f560f7daa1c9,406335d9-a74b-4914-950d-b0e34191967f


### Enrichment

In [11]:
enr_df = repository.read_enrichment(enrichment_producer_run_id)

display_df(enr_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,USM,TRD,2000,LIABILITY,CREDIT,...,130000,210000,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,USM,FIN,4000,REVENUE,CREDIT,...,410000,410000,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,USM,FIN,3000,EQUITY,CREDIT,...,310000,310000,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,CAM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,CAM,TRD,2100,LIABILITY,CREDIT,...,140000,230000,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,252c51a6-c44d-4898-ba88-009ce437ced4


### Reporting

In [12]:
rpt_df = repository.read_reporting(reporting_producer_run_id)

display_df(rpt_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,231848ad-fe2b-48b3-9436-f560f7daa1c9,acd7dfd7-903c-4291-bfe9-7447753bcbae


### Posting

In [13]:
pst_df = repository.read_posting(posting_producer_run_id)

display_df(pst_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,POSTING_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,USM,TRD,1000,ASSET,...,0E-12,90000.000000000000,80000.000000000000,10000.000000000000,0E-12,90000.000000000000,0E-12,90000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,USM,FIN,1200,ASSET,...,0E-12,20000.000000000000,15000.000000000000,5000.000000000000,0E-12,20000.000000000000,0E-12,20000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,USM,TRD,2000,LIABILITY,...,0E-12,-65000.000000000000,-55000.000000000000,0E-12,-10000.000000000000,-65000.000000000000,0E-12,-65000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,USM,FIN,4000,REVENUE,...,0E-12,-25000.000000000000,-20000.000000000000,0E-12,-5000.000000000000,-25000.000000000000,0E-12,-25000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,USM,FIN,3000,EQUITY,...,0E-12,-20000.000000000000,-20000.000000000000,0E-12,0E-12,-20000.000000000000,0E-12,-20000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,CAM,TRD,1000,ASSET,...,1000.000000000000,51000.000000000000,40000.000000000000,10000.000000000000,0E-12,50000.000000000000,1000.000000000000,51000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,CAM,TRD,2100,LIABILITY,...,-1000.000000000000,-51000.000000000000,-40000.000000000000,0E-12,-10000.000000000000,-50000.000000000000,-1000.000000000000,-51000.000000000000,231848ad-fe2b-48b3-9436-f560f7daa1c9,0cf97fc2-f09f-4de3-9907-6dcaeddec2bf


### Interface

This is the Foundry Interface output — the same rows GL reads as input for `GL / IMPORT`,
selected by `WORKFLOW_RUN_ID` + `PRODUCER_RUN_ID` lineage rather than business date/batch.

In [6]:
int_df = repository.read_interface(interface_producer_run_id)

display_df(int_df)

,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,...,POSTING_STREAM,SRC_RECORD_ID,SRC_APP_CD,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,1,1000,1100,NYC,101000,1000,...,GROSS_UP,rec-1,NFM,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,2,1000,1200,NYC,120000,1200,...,GROSS_UP,rec-2,NFM,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,3,1000,1100,NYC,210000,2000,...,GROSS_UP,rec-3,NFM,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,4,1000,1200,NYC,410000,4000,...,GROSS_UP,rec-4,NFM,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,5,1000,1200,NYC,310000,3000,...,GROSS_UP,rec-5,NFM,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-CAM-...,1,2000,2100,TOR,101000,1000,...,GROSS_UP,rec-6,NFM,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,16c36d19-d910-4ae4-8553-937e2c1068f2,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-CAM-...,2,2000,2100,TOR,230000,2100,...,GROSS_UP,rec-7,NFM,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31


## GL persistence layer

`GL / IMPORT` (`gl_run_id`) reads the Interface rows produced by `FOUNDRY / INTERFACE`
above and stamps its own execution identity onto whatever it writes — `gl.posting` and
`gl.rejection` rows carry `PRODUCER_RUN_ID = gl_run_id`, not the Interface producer's ID.

### GL Posting

In [7]:
gl_postings = gl.get_postings(gl_run_id)

display_df(gl_postings)

,GL_POSTING_ID,POSTED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,FOUNDRY_RULE_ID,POSTING_ID,POSTING_STREAM,...,BOOK_CD,SOURCE_CD,CR_DR_IND,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,66d10650-3c32-4ce7-bb61-d81905590210,2026-08-27 17:07:55.623652,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-CAM-...,1,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
1,d75f484c-d686-45df-a2e7-d11f8da4e867,2026-08-27 17:07:57.182891,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-CAM-...,2,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31
2,fe7e836b-0bc6-4be9-83c6-18ab311aefb3,2026-08-27 17:07:57.870218,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,1,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,2397aaf9-d80d-4899-a38f-399ae36cab23,2026-08-27 17:07:58.550482,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,2,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,e03362e4-866f-471e-af00-4b4bef5f62df,2026-08-27 17:07:59.211839,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,3,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,058041f4-80a8-42c2-8535-e9c92b1dbdf3,2026-08-27 17:07:59.878455,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,4,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
6,062484f6-0796-447b-a42e-0fb054eecfb9,2026-08-27 17:08:00.516475,231848ad-fe2b-48b3-9436-f560f7daa1c9,42a38c93-1d6a-4277-8bc2-d592d3b4da32,TRIAL_BALANCE,NFTB-252c51a6-c44d-4898-ba88-009ce437ced4-USM-...,5,TB-GROSS-UP,PST-260331-260331-0cf97fc2-f09f-4de3-9907-6dca...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31


### GL Rejection

A non-zero rejected count here is a normal business outcome, not an execution failure —
`GL / IMPORT` still completes as `SUCCEEDED` as long as processing itself ran cleanly.

In [12]:
gl_rejections = gl.get_rejections(gl_run_id)

display_records(gl_rejections)

""


In [8]:
print(int_df.columns)
print(gl_postings.columns)

['WORKFLOW_RUN_ID', 'PRODUCER_RUN_ID', 'DATACLASS', 'TRANSACTION_NUMBER', 'LINE_NUMBER', 'ENTITY_CD', 'DEPT_CD', 'BRANCH_CD', 'GL_ACCOUNT', 'SUB_ACCOUNT', 'AFFILIATE_CD', 'PRODUCT_CD', 'BOOK_CD', 'SOURCE_CD', 'CR_DR_IND', 'FOUNDRY_RULE_ID', 'POSTING_ID', 'POSTING_STREAM', 'SRC_RECORD_ID', 'SRC_APP_CD', 'TRANSACTION_CURRENCY', 'TRANSACTION_AMOUNT', 'ACCOUNTED_CURRENCY', 'ACCOUNTED_AMOUNT', 'FX_RATE', 'AS_OF_DATE', 'BUSINESS_DATE']
['GL_POSTING_ID', 'POSTED_AT', 'WORKFLOW_RUN_ID', 'PRODUCER_RUN_ID', 'DATACLASS', 'TRANSACTION_NUMBER', 'LINE_NUMBER', 'FOUNDRY_RULE_ID', 'POSTING_ID', 'POSTING_STREAM', 'SRC_RECORD_ID', 'SRC_APP_CD', 'ENTITY_CD', 'DEPT_CD', 'BRANCH_CD', 'GL_ACCOUNT', 'SUB_ACCOUNT', 'AFFILIATE_CD', 'PRODUCT_CD', 'BOOK_CD', 'SOURCE_CD', 'CR_DR_IND', 'TRANSACTION_CURRENCY', 'TRANSACTION_AMOUNT', 'ACCOUNTED_CURRENCY', 'ACCOUNTED_AMOUNT', 'FX_RATE', 'AS_OF_DATE', 'BUSINESS_DATE']


In [20]:
# reconciliation
recon_keys = [
    'WORKFLOW_RUN_ID',
    'AS_OF_DATE',
    "ENTITY_CD",
    "DEPT_CD",
    "BRANCH_CD",
    "GL_ACCOUNT",
    "SUB_ACCOUNT",
    "AFFILIATE_CD",
    "PRODUCT_CD",
    "BOOK_CD",
    "SOURCE_CD",
    "ACCOUNTED_CURRENCY",
]

interface_recon = (
    int_df
    .groupBy(*recon_keys)
    .agg(
        F.sum("ACCOUNTED_AMOUNT").alias("INTERFACE_BALANCE")
    )
)

gl_recon = (
    gl_postings
    .groupBy(*recon_keys)
    .agg(
        F.sum("ACCOUNTED_AMOUNT").alias("GL_BALANCE")
    )
)

recon = (
    interface_recon
    .join(gl_recon, recon_keys, "full")
    .fillna({
        "INTERFACE_BALANCE": 0,
        "GL_BALANCE": 0,
    })
    .withColumn(
        "DIFFERENCE_AMOUNT",
        F.col("INTERFACE_BALANCE") - F.col("GL_BALANCE"),
    )
)

In [21]:
display_df(interface_recon)

,WORKFLOW_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000


In [22]:
display_df(gl_recon)

,WORKFLOW_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,GL_BALANCE
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000


In [23]:
display_df(recon)

,WORKFLOW_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE,GL_BALANCE,DIFFERENCE_AMOUNT
0,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000,90000.000000000000,0E-11
1,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000,-65000.000000000000,0E-11
2,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000,20000.000000000000,0E-11
3,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000,-20000.000000000000,0E-11
4,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000,-25000.000000000000,0E-11
5,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000,38250.000000000000,0E-11
6,231848ad-fe2b-48b3-9436-f560f7daa1c9,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000,-38250.000000000000,0E-11


In [13]:
# spark.stop()